# topk-predictions — worked example 3: How many extra labels top-k covers

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `topk-predictions`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Comparing top-1 (`argmax`) against top-k reveals the 'coverage wedge': samples the top-k captures that top-1 misses. You build a top-1 hit mask and a top-k hit mask over the same labels, then count where top-k succeeds but top-1 fails. This quantifies how much a wider candidate set helps.

## Worked solution

We measure how many samples are rescued by widening from top-1 to top-3.

1. `top1 = logits.argmax(dim=1)` gives the single best class per row; `hit1 = top1 == labels` is the top-1 hit mask.
2. `top3 = logits.topk(3, dim=1).indices` gives the 3 best ids; `hit3 = (top3 == labels[:, None]).any(dim=1)` is the top-3 hit mask via broadcasting.
3. The wedge is `hit3 & ~hit1` — samples top-3 caught but top-1 missed. We sum the masks to integer counts.

The printed counts show top-3 hits are at least top-1 hits, and the wedge is exactly their gap.

In [ ]:
import torch as t

t.manual_seed(2)

def coverage_wedge(logits: t.Tensor, labels: t.Tensor) -> dict:
    top1 = logits.argmax(dim=1)
    top3 = logits.topk(3, dim=1).indices
    hit1 = (top1 == labels)
    hit3 = (top3 == labels[:, None]).any(dim=1)
    wedge = hit3 & (~hit1)
    return {'top1': int(hit1.sum()), 'top3': int(hit3.sum()), 'wedge': int(wedge.sum())}

logits = t.randn(16, 8)
labels = t.randint(0, 8, (16,))
res = coverage_wedge(logits, labels)
print(res)
print('consistent:', res['top3'] == res['top1'] + res['wedge'])